# Изучение развития игровой индустрии с 2000 по 2013 год

- Автор: Стогниева Дарья Александровна
- Дата: 10.01.2026

### Цели и задачи проекта

**Цель проекта:** знакомство с данными, проверка их корректности и проведение предобработки для получения необходимого среза данных.

**Задачи проекта:**
* Загрузка и знакомство с данными;
* Проверка ошибок в данных и их предобработка;
* Фильтрация данных;
* Категоризация данных;
* Оформление выводов.


### Описание данных

Данные  `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

* `Name` — название игры.
* `Platform` — название платформы.
* `Year of Release` — год выпуска игры.
* `Genre` — жанр игры.
* `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
* `EU sales` — продажи в Европе (в миллионах проданных копий).
* `JP sales` — продажи в Японии (в миллионах проданных копий).
* `Other sales` — продажи в других странах (в миллионах проданных копий).
* `Critic Score` — оценка критиков (от 0 до 100).
* `User Score` — оценка пользователей (от 0 до 10).
* `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

1. Предобработаем данные: приведем все названия столбцов к регистру snake case, исправим типы данных там, где это необходимо, обработаем пропуски и избавимся от дубликатов.
2. Отберем данные по времени выхода игры. Нужен период с 2000 по 2013 год включительно.
3. Категоризируем игры по оценкам пользователей и экспертов. Выделим три категории:
    * высокая оценка — с оценкой от 8 до 10 и от 80 до 100, включая правые границы интервалов.
    * средняя оценка — с оценкой от 3 до 8 и от 30 до 80, не включая правые границы интервалов.
    * низкая оценка — с оценкой от 0 до 3 и от 0 до 30, не включая правые границы интервалов.
4. Выделим топ-7 платформ по количеству игр, выпущенных за весь требуемый период.

## 1. Загрузка данных и знакомство с ними

In [1]:
# Загрузим необходимые библиотеки Python:
# В нашем случае потребуется только библиотека pandas:
import pandas as pd

In [2]:
# Выгрузим данные датасета и создадим датафрейм df:
data = pd.read_csv('/datasets/new_games.csv')
df = pd.DataFrame(data)

In [3]:
# Выведем первые 5 строк датасета:
df.head(5)

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [5]:
# Отобразим информацию о датасете с помощью метода info():
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB
None


Данные содержат 11 столбцов и 16956 строк, названия столбцов соответствуют указанным в описании. Типы данных не во всех столбцах представленны корректно: Столбцы "EU sales", "JP sales", "User Score" требуют преобразования в вещественный тип данных, а столбцы "Year of Release" и "Critic Score" - в целочисленный. В столбцах "Name", "Year of Release", "Genre", "Critic Score", "User Score" и "Rating" присутствуют пропуски. Подсчитаем количество пропусков позже.

Названия столбцов, которые содержат более 1 слова, прописаны в неудобном для работы виде и требуют переформатирования. Также, название столбца "Rating" не отображает того, какие именно данные в нем содержатся, так что необходимо уточнение в названии.

In [6]:
# Для дальнейшего анализа сохраним количество строк до предобработки данных:
initial_rows = len(df)

print(initial_rows)

16956


---

## 2. Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

In [7]:
# Выведем названия всех столбцов датафрейма:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


In [8]:
# Приведем все названия столбцов к единному стилю snake case:
columns_snake_case = ['name', 'platform', 'year_of_release', 'genre', 'NA_sales', 'EU_sales',
       'JP_sales', 'other_sales', 'critic_score', 'user_score', 'rating']

# Передаём список атрибуту columns датафрейма
df.columns = columns_snake_case

# Выводим названия столбцов датафрейма
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'NA_sales', 'EU_sales',
       'JP_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')


In [9]:
# Дабавим уточнение в название столбца "rating" - "rating_ESRB":
df = df.rename(columns={'rating': 'rating_ESRB'})
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'NA_sales', 'EU_sales',
       'JP_sales', 'other_sales', 'critic_score', 'user_score', 'rating_ESRB'],
      dtype='object')


In [10]:
# создаем копию датасета до преобразования для возможности проверить сделанные изменения после предобработки
tmp = df.copy() 
len(tmp)

16956

Ранее мы уже отметили, что cтолбцы "EU_sales", "JP_sales", "critic_score", "user_score" и "year_of_release" имеют некорректные типы данных. Возможно, данные заполнялись вручную, что могло повлиять на итоговоый тип данных в стобцах. Также, некорректные типы данных в столбцах "critic_score", "user_score" и "year_of_release" могут быть связанны с пропусками в значениях.

К тому же, столбцы "EU_sales", "JP_sales" и "user_score" содержат в себе тип данных object, хотя значения в них - числовые. Чтобы понять с чем это связанно, выведем все уникальные значения столбцов:

In [11]:
# Выводим все уникальные значения с помощью метода unique():
unique_eu_sales = df['EU_sales'].unique()
unique_jp_sales = df['JP_sales'].unique()
unique_user_score = df['user_score'].unique()

# Выводим значения:
print(f'Уникальные значения в столбце "EU_sales":')
print(unique_eu_sales)
print(f'Уникальные значения в столбце "JP_sales":')
print(unique_jp_sales)
print(f'Уникальные значения в столбце "user_score":')
print(unique_user_score)

Уникальные значения в столбце "EU_sales":
['28.96' '3.58' '12.76' '10.93' '8.89' '2.26' '9.14' '9.18' '6.94' '0.63'
 '10.95' '7.47' '6.18' '8.03' '4.89' '8.49' '9.09' '0.4' '3.75' '9.2'
 '4.46' '2.71' '3.44' '5.14' '5.49' '3.9' '5.35' '3.17' '5.09' '4.24'
 '5.04' '5.86' '3.68' '4.19' '5.73' '3.59' '4.51' '2.55' '4.02' '4.37'
 '6.31' '3.45' '2.81' '2.85' '3.49' '0.01' '3.35' '2.04' '3.07' '3.87'
 '3.0' '4.82' '3.64' '2.15' '3.69' '2.65' '2.56' '3.11' '3.14' '1.94'
 '1.95' '2.47' '2.28' '3.42' '3.63' '2.36' '1.71' '1.85' '2.79' '1.24'
 '6.12' '1.53' '3.47' '2.24' '5.01' '2.01' '1.72' '2.07' '6.42' '3.86'
 '0.45' '3.48' '1.89' '5.75' '2.17' '1.37' '2.35' '1.18' '2.11' '1.88'
 '2.83' '2.99' '2.89' '3.27' '2.22' '2.14' '1.45' '1.75' '1.04' '1.77'
 '3.02' '2.75' '2.16' '1.9' '2.59' '2.2' '4.3' '0.93' '2.53' '2.52' '1.79'
 '1.3' '2.6' '1.58' '1.2' '1.56' '1.34' '1.26' '0.83' '6.21' '2.8' '1.59'
 '1.73' '4.33' '1.83' '0.0' '2.18' '1.98' '1.47' '0.67' '1.55' '1.91'
 '0.69' '0.6' '1.93' '1.64' '

Теперь мы видим, что в столбцах "EU_sales" и "JP_sales" есть не числовое значение "unknown" и в столбце "user_score" значение "tbd". Теперь, в В столбцах "EU_sales", "JP_sales" и "user_score" мы можем сразу преобразовать тип данных в вещественный, при этом все строковые значения заменим на пропуски.

In [12]:
# Приведем типы данных столбцов "EU_sales", "JP_sales" и "user_score"  к float64 с помощью метода to_numeric():
# В случаях, когда в столбцах встретятся значения "unknown" и "tbd", заменим их на пропуски с помощью аттрибута errors='coerce':
df['EU_sales'] = pd.to_numeric(df['EU_sales'],errors='coerce')
df['JP_sales'] = pd.to_numeric(df['JP_sales'],errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'],errors='coerce')

# Выводим информацию о датафрейме:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   NA_sales         16956 non-null  float64
 5   EU_sales         16950 non-null  float64
 6   JP_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating_ESRB      10085 non-null  object 
dtypes: float64(7), object(4)
memory usage: 1.4+ MB


В стобцах "critic_score" и "year_of_release" присутствуют пропуски, которые необходимо обработать перед тем как привести их к другому типу данных. Вернемся к этому пункту после обработки пропусков.

### 2.2. Наличие пропусков в данных

In [13]:
# Считаем пропуски в датафрейме df
print(df.isna().sum()) 

name                  2
platform              0
year_of_release     275
genre                 2
NA_sales              0
EU_sales              6
JP_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating_ESRB        6871
dtype: int64


In [14]:
# Подсчитаем долю строк с пропусками:
print(df.isna().sum() / len(df)) 

name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
NA_sales           0.000000
EU_sales           0.000354
JP_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating_ESRB        0.405225
dtype: float64


Пропуски характерны для следующих столбцов:

- name: 2 пропуска(0.012%)
- year_of_release: 275 пропусков(1.62%)
- genre: 2 пропуска(0.012%)
- EU_sales: 6 пропусков(0.035%)
- JP_sales: 4 пропуска(0.024%)
- critic_score: 8714 пропусков(51.4%)
- user_score: 9268 пропусков(54.66%)
- rating_ESRB: 6871 пропусков(40.5%)

Возможные причины возникновения пропусков: 

- Ошибки при вводе данных вручную. 
- Проблемы с форматом данных при импорте из внешних источников. 
- Отсутствие информации по определённым полям.

Что можем сделать с пропусками: 

1. Заполнить пропуски(например, средним, медианым или модальным значением): 
    - в столбцах "critic_score", "user_score" и "rating_ESRB" критически много пропусков чтобы их просто удалить, заполнить их средним значением не получится исходя из особенностей данных, поэтому будем использовать значение-индикатор, например "-1". 
    - Пропуски в данных с количеством проданных копий игры в Европе("EU_sales") и Японии("JP_sales") можно заменить на среднее значение в зависимости от названия платформы и года выхода игры. 

2. Удалить строки с пропусками: в столбцах "name", "genre" и "year_of_release" количество пропусков составляет менее 2%, их не много, так что можем просто удалить их.

In [16]:
# Удаляем пропуски там, где это не критично:
df = df.dropna(subset=['name','genre','year_of_release'])

In [20]:
# Теперь, заменим пропуски в столбцах "critic_score","user_score" и "rating_ESRB" значением - индикатором:
# Будем использовать метод fillna() и значение -1 для заполнения пропусков:
df['critic_score'] = df['critic_score'].fillna(-1)
df['user_score'] = df['user_score'].fillna(-1)
df['rating_ESRB'] = df['rating_ESRB'].fillna(-1)

# Чтобы убедиться, что все сработало правильно, выведем первые 5 строк датафрейма:
df.head(5)

,name,platform,year_of_release,genre,NA_sales,EU_sales,JP_sales,other_sales,critic_score,user_score,rating_ESRB
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,-1.0,-1.0,-1
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,-1.0,-1.0,-1


Теперь, вместо пропусков NaN мы видим значения "-1.0" и "-1". В столбцах "critic_score" и "user_score" значение -1" не отображается из-за того, что они принадлежат к типу данных float64. 

Сейчас, заменим пропуски в столбцах "EU_sales" и "JP_sales" на среднее значение в зависимости от названия платформы и года выхода игры:

In [21]:
# Начнем со столбца "EU_sales":
# Создадим функцию mean_eu_sales:
def mean_eu_sales(row):
    if pd.isna(row['EU_sales']):
        # Добавим группу по названию платформы и году выхода игры:
        group_eu_sales = df[(df['platform'] == row['platform']) & 
                               (df['year_of_release'] == row['year_of_release'])]
        return group_eu_sales['EU_sales'].mean()
    else:
        return row['EU_sales']

# Применим функцию к столбцу:
df['EU_sales'] = df.apply(mean_eu_sales, axis=1)

In [22]:
# Теперь избавимся от пропусков в столбце "JP_sales":
# Создадим функцию mean_eu_sales:
def mean_jp_sales(row):
    if pd.isna(row['JP_sales']):
        # Добавим группу по названию платформы и году выхода игры и найдем среднее значение по группе:
        group_jp_sales = df[(df['platform'] == row['platform']) & 
                               (df['year_of_release'] == row['year_of_release'])]
        return group_jp_sales['JP_sales'].mean()
    else:
        return row['JP_sales']

# Применим функцию к столбцу:
df['JP_sales'] = df.apply(mean_jp_sales, axis=1)

In [23]:
# Проверим остались ли пропуски:
df.isna().sum()

name               0
platform           0
year_of_release    0
genre              0
NA_sales           0
EU_sales           0
JP_sales           0
other_sales        0
critic_score       0
user_score         0
rating_ESRB        0
dtype: int64

Пропусков нет!

Пропуски обработаны, а, значит, мы можем вернуться к приведению столбцов "critic_score" и "year_of_release" к нужному нам типу данных - int64:

In [24]:
# Воспользуемся методом .astype чтобы привести столбцы к типу int64:
df[['critic_score','year_of_release']] = df[['critic_score','year_of_release']].astype('int64')

# Выводим информацию о датафрейме:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16679 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16679 non-null  object 
 2   year_of_release  16679 non-null  int64  
 3   genre            16679 non-null  object 
 4   NA_sales         16679 non-null  float64
 5   EU_sales         16679 non-null  float64
 6   JP_sales         16679 non-null  float64
 7   other_sales      16679 non-null  float64
 8   critic_score     16679 non-null  int64  
 9   user_score       16679 non-null  float64
 10  rating_ESRB      16679 non-null  object 
dtypes: float64(5), int64(2), object(4)
memory usage: 1.5+ MB


Все пропуски обработаны и все столбцы приведены к нужному типу данных. Можно переходить к обработке дубликатов.

In [25]:
# Комментарий ревьюера
# Посмотрим какие пропуски остались
show_missing_stats(df)

'Пропусков в данных нет'

### 2.3. Явные и неявные дубликаты в данных

In [26]:
# Изучим неявные дубликаты с помощью метода unique():
unique_genre = df['genre'].unique()
unique_platform = df['platform'].unique()
unique_rating_ESRB = df['rating_ESRB'].unique()
unique_year_of_release = df['year_of_release'].unique()

# Выводим значения:
print(f'Уникальные жанры:')
print(unique_genre)
print(f'Уникальные платформы:')
print(unique_platform)
print(f'Уникальные рейтинги:')
print(unique_rating_ESRB)
print(f'Уникальные годы выпуска')
print(unique_year_of_release)

Уникальные жанры:
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
Уникальные платформы:
['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
Уникальные рейтинги:
['E' -1 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
Уникальные годы выпуска
[2006 1985 2008 2009 1996 1989 1984 2005 1999 2007 2010 2013 2004 1990
 1988 2002 2001 2011 1998 2015 2012 2014 1992 1997 1993 1994 1982 2016
 2003 1986 2000 1995 1991 1981 1987 1980 1983]


С названиями платформ и годами выпуска все хорошо: все значения уникальны. Однако, со столбцами "genre" и "rating_ESRB" ситуация другая. Видно, что в названиях жанров есть неявные дубликаты: часть значений начинаются с заглавной буквы и продолжаются строчными, другая часть написана полностью заглавными буквами. Из-за этого, например, жанр гонок записан дважды: "Racing" и "RACING". Такая же ситуация обстоит и со всеми остальными названиями жанров: они записаны по 2 раза в разных форматах.

В столбце "rating_ESRB" с первого взгляда дубликатов нет и все значения уникальны. Но есть одно "но": рейтинг "E" и рейтинг "K-A" обозначают один и тот же рейтинг - игры для всех возрастов. Дело в том, что до 1998 года рейтинг "E" был известен как Kids to Adults (K-A). Так что здесь тоже есть неявные дубликаты. 

Теперь, избавимся от всех неявных дубликатов: 

In [27]:
# Сначала приведем все названия жанров к единному формату. Допустим, пусть все значения будут написаны строчными буквами. 
# Применим метод lower():
df['genre'] = df['genre'].str.lower()

#Еще раз выведем все уникальные названия жанров:
print(df['genre'].unique())

['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']


Все значения в стобце "genre" теперь уникальны.

In [28]:
# Избавимся от неявных дубликатов в столбце "rating_ESRB".
# Заменим рейтинг "K-A" на "E":
df['rating_ESRB'] = df['rating_ESRB'].replace('K-A', 'E')

# Проверим уникальные рейтинги:
print(df['rating_ESRB'].unique())

['E' -1 'M' 'T' 'E10+' 'AO' 'EC' 'RP']


Неявных дубликатов больше нет. Можем переходить к обработке явных дубликатов.

In [29]:
# Проверим наличие явных дубликатов в датафрейме:
# Сразу посчитаем количество дублирующихся строк:
print(df.duplicated().sum())

235


235 дублирующихся строк. Удалим их:

In [30]:
# Применим метод drop_duplicates():
df = df.drop_duplicates(subset=None, keep='first', inplace=False) 

In [31]:
# Посчитаем количество дублирующихся строк:
print(f'Дублирующихся строк: {df.duplicated().sum()}')

Дублирующихся строк: 0


In [32]:
# Посчитаем общее количество строк и сохраним в переменную rows_count:
rows_count = len(df)
print(f'Всего строк: {rows_count}')

Всего строк: 16444


В этом пункте мы занимались поиском явных и неявных дубликатов.

Сначала, мы нашли неявные дубликаты в столбцах "genre" и "rating_ESRB". Каждое название жанра было записано по 2 раза, но в разных форматах. А рейтинг "Игры для всех возрастов" был записан как "E" и "K-A". Названия жанров мы привели к единному регистру, а рейтинг "K-A" заменили на "E", тем самым избавились от неявных дублкатов.

После этого, мы посчитали количество явных дубликатов: таких строк было 235. С помощью метода drop_duplicates() мы убрали все дублирующиеся строки и очистили данные для дальнейшего анализа. У нас осталость 16461 строка.

In [33]:
# Посчитаем количество строк в абсолютном и относительном значениях:
# Найдем количество удаленных строк:
deleted_rows_count = initial_rows - rows_count

print(f'Всего удаленных строк: {deleted_rows_count}')

# Найдем долю удаленных строк от общего количества:
deleted_rows_share = round((deleted_rows_count/initial_rows*100),2)

print(f'% удаленных строк: {deleted_rows_share}')

Всего удаленных строк: 512
% удаленных строк: 3.02


In [34]:
# Комментарий ревьюера
# Проверим сколько удалено строк датасета
a, b = len(tmp), len(df)
print(" Было строк в исходном датасете", a,
      '\n', "Осталось строк в датасете после обработки", b,
      '\n', "Удалено строк в датасете после обработки", a-b,
      '\n', "Процент потерь", round((a-b)/a*100, 2))

 Было строк в исходном датасете 16956 
 Осталось строк в датасете после обработки 16444 
 Удалено строк в датасете после обработки 512 
 Процент потерь 3.02


### Выводы

В ходе предобработки были выполнены следующие шаги:

1. Форматирование названий столбцов для удобной работы:
    - Все названия столбцов привели к единному стилю snake case;
    - Название столбца "Rating" заменили на "rating_ESRB";
    

2. Приведение столбцов к правильным типам данных:
    - Сначала в столбцах "EU_sales", "JP_sales" и "user_score" выявили причину не правильного типа данных - строковые значения "unknown" и "tbd", затем привели столбцы к типу данных float64, заменив строковые значения на пропуски;
    - После обработки пропусков столбцы "critic_score" и "year_of_release" привели к целочисленному типу данных int64;
    

3. Обработка пропусков:
    - Были обнаружены и обработаны пропущенные значения в столбцах  "name", "year_of_release", "genre", "EU_sales", "JP_sales", "critic_score", "user_score" и "rating_ESRB".
    - Пропуски в столбцах "name", "year_of_release" и "genre" удалены, а в столбцах "critic_score", "user_score" и "rating_ESRB" заменены на значение-индикатор "-1". В столбцах "EU_sales" и "JP_sales" пропуски заполнили средним значением для игр выпущенных на той же платформе и в том же году.
    
    
4. Удаление дубликатов:
    - Выявлены и обработаны неявные дубликаты в столбцах "genre" и "rating_ESRB";
    - Выявлены и удалены дубликаты данных в количестве 235 строк;

#### Результаты предобработки:

- Общее количество строк до предобработки: 16956.
- Количество удалённых строк: 512 (абсолютное значение) и 3.02% (относительное значение).
- Всего осталось строк после предобработки: 16444.

В результате данные стали более структурированными и чистыми, и готовы для дальнейшего анализа.

---

## 3. Фильтрация данных

Нам нужно изучить историю продаж игр в начале XXI века, то есть в период с 2000 по 2013 год включительно. Отберем данные по этому показателю:

In [35]:
# Отфильтруем данные и сохраним их в новом датафрейме df_actual:
df_actual = df.loc[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [36]:
# Комментарий ревьюера
df_actual.info(), df_actual.year_of_release.sort_values().unique()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12781 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12781 non-null  object 
 1   platform         12781 non-null  object 
 2   year_of_release  12781 non-null  int64  
 3   genre            12781 non-null  object 
 4   NA_sales         12781 non-null  float64
 5   EU_sales         12781 non-null  float64
 6   JP_sales         12781 non-null  float64
 7   other_sales      12781 non-null  float64
 8   critic_score     12781 non-null  int64  
 9   user_score       12781 non-null  float64
 10  rating_ESRB      12781 non-null  object 
dtypes: float64(5), int64(2), object(4)
memory usage: 1.2+ MB


(None,
 array([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
        2011, 2012, 2013]))

---

## 4. Категоризация данных

Для начала, нам необходимо разделить все игры на 3 категории по оценкам пользователей: 

- высокая оценка (от 8 до 10)
- средняя оценка (от 3 до 8)
- низкая оценка (от 0 до 3)

In [37]:
# С помощью метода pd.cut() разделим игры на категории и сохраним значения в новом столбце 'user_score_group':
df_actual.loc[:,'user_score_group'] = pd.cut(df_actual['user_score'], bins=[0, 3, 8, 10], labels=["Низкая оценка", "Средняя оценка", "Высокая оценка"],right=False)

Теперь, разделим все игры на 3 категории по оценкам критиков:

- высокая оценка (от 80 до 100)
- средняя оценка (от 30 до 80)
- низкая оценка (от 0 до 30)

In [38]:
# Воспользуемся тем же методом pd.cut() и сохраним значения в новом столбце 'critic_score_group':
df_actual.loc[:,'ctiric_score_group'] = pd.cut(df_actual.loc[:,'critic_score'], bins=[0, 30, 80, 100], labels=["Низкая оценка", "Средняя оценка", "Высокая оценка"],right=False)

Теперь проверим результат: сгруппируем данные по категориям и посчитаем количество игр в каждой категории:

In [39]:
# Группируем данные по категориям по оценке пользователей с помощью метода groupby():
grouped_user_score = df_actual.groupby('user_score_group', as_index=False)['name'].count()

# Выводим результат:
print(f'Оценки пользователей:')
print(grouped_user_score)

Оценки пользователей:
  user_score_group  name
0    Низкая оценка   116
1   Средняя оценка  4081
2   Высокая оценка  2286


In [40]:
# Группируем данные по категориям по оценке критиков с помощью метода groupby():
grouped_critic_score = df_actual.groupby('ctiric_score_group', as_index=False)['name'].count()

# Выводим результат:
print(f'Оценки критиков:')
print(grouped_critic_score)

Оценки критиков:
  ctiric_score_group  name
0      Низкая оценка    55
1     Средняя оценка  5422
2     Высокая оценка  1692


Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [41]:
# Сгруппируем данные по платформам и посчитаем количество выпущенных игр на каждой из них:
grouped_by_platform = df_actual.groupby('platform', as_index=False)['name'].count()

In [42]:
# Отсортируем данные по количеству выпущенных игр по убыванию с помощью метода sort_values():
platforms_sorted = grouped_by_platform.sort_values(by='name', ascending=False)

# Превратившиеся в индексы группировочные столбцы преобразуем обратно в столбцы и уберем дополнительный столбец с индексами:
platforms_sorted = platforms_sorted.reset_index(drop=True)

# Начнем нумерацию индексов с 1:
platforms_sorted.index = platforms_sorted.index + 1

# Выведем результат:
print(platforms_sorted.head(7))

  platform  name
1      PS2  2127
2       DS  2120
3      Wii  1275
4      PSP  1180
5     X360  1121
6      PS3  1087
7      GBA   811


---

## 5. Итоговый вывод

В ходе работы над проектом были выполнены следующие шаги:

1. Предобработка данных:
    1. Форматирование названий столбцов для удобной работы:
        - Все названия столбцов привели к единному стилю snake case;
        - Название столбца "Rating" заменили на "rating_ESRB";


    2. Приведение столбцов к правильным типам данных:
    - Сначала в столбцах "EU_sales", "JP_sales" и "user_score" выявили причину не правильного типа данных - строковые значения "unknown" и "tbd", затем привели столбцы к типу данных float64, заменив строковые значения на пропуски;
    - После обработки пропусков столбцы "critic_score" и "year_of_release" привели к целочисленному типу данных int64;
    

    3. Обработка пропусков:
    - Были обнаружены и обработаны пропущенные значения в столбцах  "name", "year_of_release", "genre", "EU_sales", "JP_sales", "critic_score", "user_score" и "rating_ESRB".
    - Пропуски в столбцах "name", "year_of_release" и "genre" удалены, а в столбцах "critic_score", "user_score" и "rating_ESRB" заменены на значение-индикатор "-1". В столбцах "EU_sales" и "JP_sales" пропуски заполнили средним значением для игр выпущенных на той же платформе и в том же году.
    
    
    4. Удаление дубликатов:
    - Выявлены и обработаны неявные дубликаты в столбцах "genre" и "rating_ESRB";
    - Выявлены и удалены дубликаты данных в количестве 235 строк;


**Результаты предобработки:**


Общее количество строк до предобработки: 16956.
Количество удалённых строк: 512 (абсолютное значение) и 3.02% (относительное значение).
Всего осталось строк после предобработки: 16444.

2. Фильтрация данных:

- Выполнен срез данных, включающий только записи с играми, выпущенные в период с 2000 по 2013 год.

3. Категоризация данных:

- Все игры были разделены на 3 категории: по оценкам пользователей и по оценкам критиков. В исходный датасет не были добавлены новые поля, но был создан новый датафрейм "df_actual", куда, помимо столбцов датафрейма df, были добавлены поля "grouped_user_score" и "grouped_critic_score", которые позволяют группировать оценки пользователей и критиков по категориям «Низкая оценка», «Средняя оценка», «Высокая оценка».

- Также были выделены топ 7 платформ по количеству выпущенных игр: 

1.       PS2  - 2127

2.        DS  - 2122

3.       Wii - 1277

4.       PSP - 1181

5.      X360 - 1119

6.       PS3 - 1089

7.       GBA -  811

**Основной вывод:**
После проведения предобработки данных датасет может быть использован для дальнейшего анализа. Данные были разделены на категории, сгруппированы и отфильтрованы по необходимым условиям, что может быть полезно для принятия решений по привлечению новой аудитории и подготовки статьи о развитии индустрии игр в начале XXI века.